In [30]:
import os
from dotenv import load_dotenv
import openai
import requests
import re

load_dotenv()

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"


In [31]:
def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    return response.json()

def get_movie_details(id: int):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return response.json()

def get_movie_credits(id: int):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    return response.json()

In [32]:
PROMPT = """
You are a movie expert agent.

You can use the following functions:

1. get_popular_movies()
2. get_movie_details(id)
3. get_movie_credits(id)

IMPORTANT:
- Only respond with the function call.
- Do NOT explain.
- Do NOT add extra text.

Examples:
get_popular_movies()
get_movie_details(550)
get_movie_credits(550)
"""

In [33]:
tools={
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits      
}

def parse_tool_call(text):
    text = text.strip()

    match = re.fullmatch(r"([a-zA-Z_]\w*)\((.*?)\)", text)
    if not match:
        raise ValueError("Invalid function format")

    func_name = match.group(1)
    args = match.group(2)

    if func_name not in tools:
        raise ValueError("Function not allowed")

    if args == "":
        return func_name, []

    return func_name, [int(args)]
    

In [34]:
from annotated_types import T


def run_movie_agent():
    user_input = input("Enter your query: ")
    messages = [
        {"role": "system", "content": PROMPT},
        {"role": "user", "content": user_input}
    ]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )

    function_call_text = response.choices[0].message.content.strip()
    print("LLM chose:", function_call_text)

    func_name, args = parse_tool_call(function_call_text)
    result = tools[func_name](*args)
    
    print("\n📦 API Result:")
    print(result)
        



In [35]:
run_movie_agent()

LLM chose: get_movie_details(550)

📦 API Result:
{'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/c6OLXfKAk5BKeR6broC8pYiCquX.jpg', 'belongs_to_collection': None, 'budget': 63000000, 'genres': [{'id': 18, 'name': 'Drama'}, {'id': 53, 'name': 'Thriller'}], 'homepage': 'http://www.foxmovies.com/movies/fight-club', 'id': 550, 'imdb_id': 'tt0137523', 'origin_country': ['US'], 'original_language': 'en', 'original_title': 'Fight Club', 'overview': 'A ticking-time-bomb insomniac and a slippery soap salesman channel primal male aggression into a shocking new form of therapy. Their concept catches on, with underground "fight clubs" forming in every town, until an eccentric gets in the way and ignites an out-of-control spiral toward oblivion.', 'popularity': 31.8105, 'poster_path': 'https://image.tmdb.org/t/p/w780/pB8BM7pdSp6B6Ih7QZ4DrQ3PmJK.jpg', 'production_companies': [{'id': 711, 'logo_path': 'https://image.tmdb.org/t/p/w300/tEiIH5QesdheJmDAqQwvtN60727.png', 'name': 'Fox 2